# 🚗💨 CO2 Emission by Vehicles — Prediction System
## Notebook 2: Full Dataset — 10-Fold Cross-Validation (**Main Submission**)

---
**Course:** AIC354 — Machine Learning Fundamentals  
**Assignment:** 3 — Applying, Evaluating & Comparing Multiple ML Algorithms  
**Dataset:** CO2 Emissions Canada (7,385 vehicle records)  

---
### 🤖 Algorithms Compared (Different from Car Price System)

| # | Algorithm | Category | Key Hyperparameters |
|---|-----------|----------|---------------------|
| 1 | **Ridge Regression** | Linear / Regularized | `alpha=1.0` |
| 2 | **Lasso Regression** | Linear / Regularized | `alpha=0.01` |
| 3 | **Support Vector Regression** | Kernel-Based | `kernel=rbf`, `C=100` |
| 4 | **Extra Trees Regressor** | Ensemble (Bagging) | `n_estimators=200` |
| 5 | **XGBoost Regressor** | Ensemble (Boosting) | `n_estimators=200`, `lr=0.05` |

---
> **Evaluation Metric:** RMSE (Root Mean Squared Error) — lower is better  
> **Cross-Validation:** 10-Fold on **full dataset** (7,385 records)  
> **Platform:** Google Colab  

---
> ⚠️ **Deployment Link:** *(Add your cloud deployment link here after deployment)*  
> `https://your-app.render.com` or `https://your-app.railway.app`


In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 1: Install Required Libraries
# ══════════════════════════════════════════════════════════════
!pip install xgboost tqdm --quiet
print('✅ Dependencies installed.')

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 2: Import Libraries
# ══════════════════════════════════════════════════════════════
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
import time
import pickle
import os

from sklearn.linear_model import Ridge, Lasso
from sklearn.svm import SVR
from sklearn.ensemble import ExtraTreesRegressor
from xgboost import XGBRegressor

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

from google.colab import files
from tqdm.notebook import tqdm
from IPython.display import display, HTML

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams['figure.dpi'] = 130
plt.rcParams['font.family'] = 'DejaVu Sans'

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print('✅ All libraries imported successfully!')
print(f'   NumPy     : {np.__version__}')
print(f'   Pandas    : {pd.__version__}')
print(f'   Sklearn   : installed')

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 3: Upload & Load Dataset
# ══════════════════════════════════════════════════════════════
print('📂 Please upload: CO2 Emissions_Canada.csv')
uploaded = files.upload()
filename = list(uploaded.keys())[0]
df = pd.read_csv(filename)

print(f'\n✅ Dataset loaded successfully!')
print(f'   Shape   : {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'   Columns : {list(df.columns)}')

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 4: Exploratory Data Analysis — Overview
# ══════════════════════════════════════════════════════════════
print('=' * 70)
print('  SECTION 1: EXPLORATORY DATA ANALYSIS (EDA)')
print('=' * 70)

print('\n📋 First 5 Rows:')
display(df.head())

print('\n📊 Data Types & Non-Null Counts:')
print(df.info())

print('\n🔍 Missing Values:')
print(df.isnull().sum())

print('\n📈 Descriptive Statistics:')
display(df.describe())

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 5: Exploratory Data Analysis — Visualizations
# ══════════════════════════════════════════════════════════════
TARGET = 'CO2 Emissions(g/km)'

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('EDA — CO2 Emissions Dataset', fontsize=15, fontweight='bold')

# 1. Target distribution
axes[0, 0].hist(df[TARGET], bins=40, color='#3498db', edgecolor='white', alpha=0.85)
axes[0, 0].set_xlabel('CO2 Emissions (g/km)', fontsize=11)
axes[0, 0].set_ylabel('Frequency', fontsize=11)
axes[0, 0].set_title('Distribution of CO2 Emissions', fontsize=12)
axes[0, 0].axvline(df[TARGET].mean(), color='red', linestyle='--', label=f'Mean = {df[TARGET].mean():.0f}')
axes[0, 0].legend()

# 2. Engine Size vs CO2
axes[0, 1].scatter(df['Engine Size(L)'], df[TARGET],
                   alpha=0.3, s=10, color='#e74c3c')
axes[0, 1].set_xlabel('Engine Size (L)', fontsize=11)
axes[0, 1].set_ylabel('CO2 Emissions (g/km)', fontsize=11)
axes[0, 1].set_title('Engine Size vs CO2 Emissions', fontsize=12)

# 3. Cylinders vs CO2
cyl_means = df.groupby('Cylinders')[TARGET].mean().reset_index()
axes[1, 0].bar(cyl_means['Cylinders'].astype(str), cyl_means[TARGET],
               color='#2ecc71', edgecolor='white')
axes[1, 0].set_xlabel('Number of Cylinders', fontsize=11)
axes[1, 0].set_ylabel('Mean CO2 Emissions (g/km)', fontsize=11)
axes[1, 0].set_title('Mean CO2 by Number of Cylinders', fontsize=12)

# 4. Fuel Type vs CO2
fuel_means = df.groupby('Fuel Type')[TARGET].mean().reset_index().sort_values(TARGET, ascending=False)
axes[1, 1].barh(fuel_means['Fuel Type'], fuel_means[TARGET],
                color='#9b59b6', edgecolor='white')
axes[1, 1].set_xlabel('Mean CO2 Emissions (g/km)', fontsize=11)
axes[1, 1].set_title('Mean CO2 by Fuel Type', fontsize=12)
axes[1, 1].invert_yaxis()

plt.tight_layout()
plt.savefig('eda_plots.png', bbox_inches='tight', dpi=150)
plt.show()
print('\n📊 EDA charts saved as eda_plots.png')

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 6: Correlation Heatmap
# ══════════════════════════════════════════════════════════════
num_df = df.select_dtypes(include=[np.number])
plt.figure(figsize=(10, 7))
corr = num_df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f',
            cmap='coolwarm', linewidths=0.5,
            cbar_kws={'shrink': 0.8})
plt.title('Correlation Heatmap — Numerical Features', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', bbox_inches='tight', dpi=150)
plt.show()
print('\n🔥 Correlation heatmap saved.')

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 7: Feature Engineering & Preprocessing
# ══════════════════════════════════════════════════════════════
print('=' * 70)
print('  SECTION 2: PREPROCESSING & FEATURE ENGINEERING')
print('=' * 70)

TARGET   = 'CO2 Emissions(g/km)'
DROP     = ['Make', 'Model']   # Drop high-cardinality ID columns
CAT_COLS = ['Vehicle Class', 'Transmission', 'Fuel Type']

df_proc = df.drop(columns=DROP).copy()

# Label-encode categorical features
le_dict = {}
for col in CAT_COLS:
    le = LabelEncoder()
    df_proc[col] = le.fit_transform(df_proc[col].astype(str))
    le_dict[col] = le

# Feature matrix and target
X = df_proc.drop(columns=[TARGET])
y = df_proc[TARGET]

print(f'\n✅ Preprocessing done!')
print(f'   Total samples  : {X.shape[0]:,}')
print(f'   Total features : {X.shape[1]}')
print(f'   Features       : {list(X.columns)}')
print(f'   Target range   : {y.min()} – {y.max()} g/km')
print(f'   Target mean    : {y.mean():.2f} g/km')

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 8: Define the 5 ML Algorithms (with Pipelines)
# ══════════════════════════════════════════════════════════════
print('=' * 70)
print('  SECTION 3: MODEL DEFINITION')
print('=' * 70)

models = {
    '① Ridge Regression'      : Pipeline([
        ('scaler', StandardScaler()),
        ('model',  Ridge(alpha=1.0))
    ]),
    '② Lasso Regression'      : Pipeline([
        ('scaler', StandardScaler()),
        ('model',  Lasso(alpha=0.01, max_iter=10000))
    ]),
    '③ SVR (RBF Kernel)'      : Pipeline([
        ('scaler', StandardScaler()),
        ('model',  SVR(kernel='rbf', C=100, gamma='scale', epsilon=0.1))
    ]),
    '④ Extra Trees Regressor' : ExtraTreesRegressor(
        n_estimators=200, max_features='sqrt',
        random_state=RANDOM_STATE, n_jobs=-1
    ),
    '⑤ XGBoost Regressor'     : XGBRegressor(
        n_estimators=200, learning_rate=0.05,
        max_depth=6, subsample=0.8,
        colsample_bytree=0.8, reg_alpha=0.1,
        random_state=RANDOM_STATE, verbosity=0,
        eval_metric='rmse', n_jobs=-1
    ),
}

for name in models:
    print(f'   {name}')
print(f'\n✅ {len(models)} models ready for 10-Fold CV!')

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 9: 10-Fold Cross-Validation — Full Dataset
# ══════════════════════════════════════════════════════════════
print('=' * 70)
print('  SECTION 4: 10-FOLD CROSS-VALIDATION ON FULL DATASET')
print('=' * 70)

K_FOLDS = 10
kf = KFold(n_splits=K_FOLDS, shuffle=True, random_state=RANDOM_STATE)

results = {}
total_start = time.time()

print(f'\n🔁 Running {K_FOLDS}-Fold Cross-Validation on {X.shape[0]:,} instances...\n')

for name, model in tqdm(models.items(), desc='📊 Training Models', unit='model',
                        bar_format='{l_bar}{bar:30}{r_bar}'):
    start = time.time()

    # RMSE (via neg MSE)
    neg_mse = cross_val_score(model, X, y, cv=kf,
                              scoring='neg_mean_squared_error', n_jobs=-1)
    rmse_scores = np.sqrt(-neg_mse)

    # R² scores
    r2_scores = cross_val_score(model, X, y, cv=kf,
                                scoring='r2', n_jobs=-1)

    elapsed = time.time() - start

    results[name] = {
        'RMSE Mean' : round(rmse_scores.mean(), 4),
        'RMSE Std'  : round(rmse_scores.std(),  4),
        'R² Mean'   : round(r2_scores.mean(),   4),
        'R² Std'    : round(r2_scores.std(),    4),
        'Time (s)'  : round(elapsed, 2),
        'All RMSE'  : rmse_scores,
    }

    clean_name = name.split(' ', 1)[1]  # Strip emoji for display
    print(f'  ✔ {clean_name:<28} RMSE: {rmse_scores.mean():.4f} ± {rmse_scores.std():.4f}'
          f'  R²: {r2_scores.mean():.4f}  ({elapsed:.1f}s)')

total_elapsed = time.time() - total_start
print(f'\n⏱ Total training time: {total_elapsed:.1f}s')
print('\n✅ 10-Fold Cross-Validation complete!')

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 10: Results Comparison Table
# ══════════════════════════════════════════════════════════════
print('=' * 70)
print('  SECTION 5: RESULTS COMPARISON TABLE')
print('=' * 70)

summary = {}
for name, r in results.items():
    clean_name = name.split(' ', 1)[1]
    summary[clean_name] = {
        'RMSE Mean'  : r['RMSE Mean'],
        'RMSE Std'   : r['RMSE Std'],
        'R² Mean'    : r['R² Mean'],
        'Time (s)'   : r['Time (s)'],
    }

results_df = pd.DataFrame(summary).T.reset_index()
results_df.columns = ['Model', 'RMSE Mean', 'RMSE Std', 'R² Mean', 'Time (s)']
results_df = results_df.sort_values('RMSE Mean').reset_index(drop=True)
results_df.insert(0, 'Rank', range(1, len(results_df) + 1))

# Style the best row
def highlight_best(s):
    return ['background-color: #d4edda; font-weight: bold' if i == 0
            else '' for i in range(len(s))]

print('\n📋 MODEL COMPARISON — 10-Fold CV on Full Dataset (7,385 instances)\n')
display(results_df.style.apply(highlight_best).format({
    'RMSE Mean': '{:.4f}',
    'RMSE Std': '{:.4f}',
    'R² Mean': '{:.4f}',
    'Time (s)': '{:.2f}s'
}))

best = results_df.iloc[0]
worst = results_df.iloc[-1]
print(f'\n🏆 Best  Model : {best["Model"]}  (RMSE = {best["RMSE Mean"]:.4f}, R² = {best["R² Mean"]:.4f})')
print(f'📉 Worst Model : {worst["Model"]}  (RMSE = {worst["RMSE Mean"]:.4f}, R² = {worst["R² Mean"]:.4f})')

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 11: Comparison Charts
# ══════════════════════════════════════════════════════════════
print('=' * 70)
print('  SECTION 6: MODEL COMPARISON CHARTS')
print('=' * 70)

PALETTE = ['#2ecc71', '#3498db', '#9b59b6', '#e67e22', '#e74c3c']
colors  = [PALETTE[0]] + PALETTE[1:len(results_df)]

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('CO2 Emission Prediction — 10-Fold CV Model Comparison\n'
             'Full Dataset (7,385 Vehicle Records)',
             fontsize=14, fontweight='bold', y=1.01)

model_names = results_df['Model'].tolist()

# ── Chart 1: RMSE Mean Bar Chart ──────────────────────────────
bars = axes[0, 0].barh(model_names, results_df['RMSE Mean'],
                        xerr=results_df['RMSE Std'],
                        color=colors, edgecolor='white', capsize=4)
axes[0, 0].set_xlabel('RMSE Mean (g/km)', fontsize=11)
axes[0, 0].set_title('RMSE Mean with Std Dev (10-Fold CV)', fontsize=12, fontweight='bold')
axes[0, 0].invert_yaxis()
for i, (val, std) in enumerate(zip(results_df['RMSE Mean'], results_df['RMSE Std'])):
    axes[0, 0].text(val + std + 0.3, i, f'{val:.2f}', va='center', fontsize=9)
gold_patch = mpatches.Patch(color='#2ecc71', label='Best Model')
axes[0, 0].legend(handles=[gold_patch])

# ── Chart 2: R² Mean Bar Chart ────────────────────────────────
axes[0, 1].barh(model_names, results_df['R² Mean'],
                color=colors, edgecolor='white')
axes[0, 1].set_xlabel('R² Score (higher = better)', fontsize=11)
axes[0, 1].set_title('R² Score Comparison', fontsize=12, fontweight='bold')
axes[0, 1].invert_yaxis()
for i, val in enumerate(results_df['R² Mean']):
    axes[0, 1].text(val + 0.001, i, f'{val:.4f}', va='center', fontsize=9)

# ── Chart 3: Box Plot of RMSE across folds ────────────────────
fold_data  = [results[name]['All RMSE'] for name in models]
clean_labs = [n.split(' ', 1)[1] for n in models]
bp = axes[1, 0].boxplot(fold_data, patch_artist=True, vert=True,
                         medianprops=dict(color='black', linewidth=2))
for patch, color in zip(bp['boxes'], PALETTE):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[1, 0].set_xticklabels([l.replace(' ', '\n') for l in clean_labs],
                             fontsize=8)
axes[1, 0].set_ylabel('RMSE per Fold (g/km)', fontsize=11)
axes[1, 0].set_title('RMSE Distribution across 10 Folds', fontsize=12, fontweight='bold')

# ── Chart 4: Training Time ────────────────────────────────────
axes[1, 1].bar(model_names, results_df['Time (s)'],
               color='#34495e', edgecolor='white')
axes[1, 1].set_ylabel('Training Time (seconds)', fontsize=11)
axes[1, 1].set_title('Training Time Comparison', fontsize=12, fontweight='bold')
axes[1, 1].tick_params(axis='x', rotation=20)
for i, val in enumerate(results_df['Time (s)']):
    axes[1, 1].text(i, val + 0.1, f'{val:.1f}s', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('comparison_notebook2.png', bbox_inches='tight', dpi=150)
plt.show()
print('\n📊 Charts saved as comparison_notebook2.png')

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 12: Train Best Model on Full Data & Save
# ══════════════════════════════════════════════════════════════
print('=' * 70)
print('  SECTION 7: TRAIN BEST MODEL ON FULL DATASET & SAVE')
print('=' * 70)

best_model_name = best['Model']
print(f'\n🏆 Best model identified: {best_model_name}')

# Find the model object by matching clean name
best_model_obj = None
for name, m in models.items():
    if name.split(' ', 1)[1] == best_model_name:
        best_model_obj = m
        break

print('\n🔄 Training best model on full dataset...')
best_model_obj.fit(X, y)

# Save to pickle
MODEL_PATH = 'best_model.pkl'
save_bundle = {
    'model'    : best_model_obj,
    'le_dict'  : le_dict,
    'features' : list(X.columns),
    'target'   : TARGET,
    'model_name': best_model_name,
}

with open(MODEL_PATH, 'wb') as f:
    pickle.dump(save_bundle, f)

print(f'\n✅ Best model saved to: {MODEL_PATH}')
print(f'   Model type   : {type(best_model_obj).__name__}')
print(f'   File size    : {os.path.getsize(MODEL_PATH) / 1024:.1f} KB')

# Download the model file
files.download(MODEL_PATH)
print('\n📥 Model file downloaded! Place it in the models/ folder.')

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 13: Feature Importance (if supported)
# ══════════════════════════════════════════════════════════════
print('=' * 70)
print('  SECTION 8: FEATURE IMPORTANCE')
print('=' * 70)

def get_feature_importance(model, feature_names):
    """Extract feature importances from a model or pipeline."""
    m = model
    if hasattr(model, 'named_steps'):
        m = model.named_steps.get('model', model)
    if hasattr(m, 'feature_importances_'):
        return m.feature_importances_
    elif hasattr(m, 'coef_'):
        return np.abs(m.coef_)
    return None

feature_names = list(X.columns)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Feature Importance — Tree-Based Models', fontsize=13, fontweight='bold')

et_model = models['④ Extra Trees Regressor']
xgb_model = models['⑤ XGBoost Regressor']

for ax, model, title in [
    (axes[0], et_model,  'Extra Trees Regressor'),
    (axes[1], xgb_model, 'XGBoost Regressor')
]:
    imp = get_feature_importance(model, feature_names)
    if imp is not None:
        fi_df = pd.DataFrame({'Feature': feature_names, 'Importance': imp})
        fi_df = fi_df.sort_values('Importance', ascending=True)
        ax.barh(fi_df['Feature'], fi_df['Importance'],
                color='#3498db', edgecolor='white')
        ax.set_xlabel('Importance Score', fontsize=11)
        ax.set_title(title, fontsize=12)
    else:
        ax.text(0.5, 0.5, 'Not Available', ha='center', va='center',
                transform=ax.transAxes, fontsize=12)

plt.tight_layout()
plt.savefig('feature_importance.png', bbox_inches='tight', dpi=150)
plt.show()
print('\n📊 Feature importance chart saved.')

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 14: Final Summary Report
# ══════════════════════════════════════════════════════════════
print('\n' + '═' * 70)
print('  FINAL SUMMARY REPORT — CO2 EMISSION PREDICTION SYSTEM')
print('═' * 70)
print(f'  Dataset    : CO2 Emissions Canada')
print(f'  Records    : {X.shape[0]:,} vehicle instances')
print(f'  Features   : {X.shape[1]} input features')
print(f'  Target     : CO2 Emissions (g/km)')
print(f'  CV Folds   : 10')
print(f'  Metric     : RMSE (Root Mean Squared Error)')
print('─' * 70)
print('  ALGORITHM RESULTS:')
print('─' * 70)
for _, row in results_df.iterrows():
    badge = '🥇' if row['Rank'] == 1 else ('🥈' if row['Rank'] == 2 else
             '🥉' if row['Rank'] == 3 else f'  {int(row["Rank"])}.')
    print(f'  {badge} {row["Model"]:<28} RMSE={row["RMSE Mean"]:.4f}  R²={row["R² Mean"]:.4f}')
print('─' * 70)
print(f'  🏆 WINNER  : {best["Model"]}')
print(f'     RMSE    : {best["RMSE Mean"]:.4f} g/km')
print(f'     R²      : {best["R² Mean"]:.4f}')
print('═' * 70)
print()
print('  Deployment: See webapp/app.py for the Flask web application.')
print('  GitHub    : https://github.com/YOUR_USERNAME/co2-emission-by-vehicles-prediction-ai')
print('  Live URL  : [Add your deployed app link here]')
print('═' * 70)